# Reward design and next-best offer

**Track:** Portfolio. **Level:** Practitioner. **CPU:** yes. **Keys:** no.

## Goals

- Simulate a contextual-bandit next-best-offer problem with a known reward.
- Compare a click reward with a margin reward on the same contexts.
- Learn a policy with LinUCB.
- Evaluate a new policy from logged data with inverse propensity scoring and doubly robust estimation.

## Sources and license

Original notebook. LinUCB is Li, Chu, Langford, and Schapire, *A Contextual-Bandit Approach to Personalized News Article Recommendation* (2010). Doubly robust policy evaluation is Dudík, Langford, and Li (ICML 2011). The logging-policy check is the Horvitz-Thompson identity: when the target probabilities equal the logging propensities, the estimate equals the mean logged reward. No paper code is copied.

## Concepts

Next-best offer is a choice of one action given a customer context. The reward you log becomes the policy you learn. A click reward prefers the offer people accept. A margin reward prefers the offer that is worth accepting. Offline evaluation matters because you cannot rerun the last quarter. IPS reweights logged rows by how rarely the logging policy took the action the new policy wants. Doubly robust estimation adds a reward model and reweights only the residual, which lowers variance when the model is decent and stays unbiased if the propensities are right.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'pandas', 'scikit-learn', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

from labs._shared.seed import seed_everything

seed_everything(0)
rng = np.random.default_rng(0)


## Guided exercise: the offer simulator

Three offers: a discount, a premium plan, and no offer. Context is four standard-normal features plus a bias. Expected margin is linear in the context, and the discount's expected margin falls as the first feature rises while the premium's rises. Expected click does the opposite of what margin wants: the discount is usually clicked, and it is the worse business outcome on average. Rewards are the expectation plus Gaussian noise.


In [ ]:
N = 4000
D = 5
X_RAW = rng.normal(size=(N, D - 1))
X = np.column_stack([X_RAW, np.ones(N)])

def expected_margin(x: np.ndarray, arm: int) -> float:
    # Arm 1 has the higher average margin. Arm 0 wins only when x[0] is low.
    if arm == 0:
        return 0.2 - 0.8 * x[0]
    if arm == 1:
        return 0.9 + 0.8 * x[0]
    return 0.15

def expected_click(x: np.ndarray, arm: int) -> float:
    if arm == 0:
        return 0.75
    if arm == 1:
        return 0.22
    return 0.08

ARMS = (0, 1, 2)
MARGIN = np.array([[expected_margin(x, a) for a in ARMS] for x in X])
CLICK = np.array([[expected_click(x, a) for a in ARMS] for x in X])
print("mean expected margin by arm", MARGIN.mean(axis=0).round(3))
print("mean expected click by arm", CLICK.mean(axis=0).round(3))
assert MARGIN.mean(axis=0)[1] > MARGIN.mean(axis=0)[0]
assert CLICK.mean(axis=0)[0] > CLICK.mean(axis=0)[1]


## Guided exercise: LinUCB

Each arm keeps `A` and `b` for ridge regression, and the score is the predicted reward plus an uncertainty bonus. The bonus is `alpha * sqrt(x^T A^{-1} x)`. The policy is trained on the first 3,000 rounds and frozen for the last 1,000.


In [ ]:
class LinUCB:
    def __init__(self, d: int, n_arms: int, alpha: float = 0.6, l2: float = 1.0):
        self.alpha = alpha
        self.A = [l2 * np.eye(d) for _ in range(n_arms)]
        self.b = [np.zeros(d) for _ in range(n_arms)]

    def scores(self, x: np.ndarray) -> np.ndarray:
        values = []
        for arm in range(len(self.A)):
            cov_inv = np.linalg.inv(self.A[arm])
            theta = cov_inv @ self.b[arm]
            bonus = self.alpha * math.sqrt(float(x @ cov_inv @ x))
            values.append(float(theta @ x + bonus))
        return np.array(values)

    def choose(self, x: np.ndarray) -> int:
        return int(np.argmax(self.scores(x)))

    def update(self, arm: int, x: np.ndarray, reward: float) -> None:
        self.A[arm] += np.outer(x, x)
        self.b[arm] += reward * x

def rollout(expected: np.ndarray, train_end: int = 3000, alpha: float = 0.6):
    policy = LinUCB(D, len(ARMS), alpha=alpha)
    chosen = np.zeros(N, dtype=int)
    reward = np.zeros(N)
    for t in range(N):
        arm = policy.choose(X[t]) if t < train_end else policy.choose(X[t])
        # Freeze learning after train_end so the test slice is on-policy evaluation.
        noise = float(rng.normal(0.0, 0.15))
        reward[t] = float(expected[t, arm] + noise)
        chosen[t] = arm
        if t < train_end:
            policy.update(arm, X[t], reward[t])
    return policy, chosen, reward

margin_policy, margin_arms, margin_reward = rollout(MARGIN)
click_policy, click_arms, click_reward = rollout(CLICK)
random_arms = rng.integers(0, len(ARMS), size=N)
random_margin = np.array([MARGIN[t, random_arms[t]] + float(rng.normal(0, 0.15)) for t in range(N)])

test = slice(3000, None)
summary = pd.DataFrame([
    {"policy": "LinUCB margin", "test margin": MARGIN[test, margin_arms[test]].mean(), "test click": CLICK[test, margin_arms[test]].mean()},
    {"policy": "LinUCB click", "test margin": MARGIN[test, click_arms[test]].mean(), "test click": CLICK[test, click_arms[test]].mean()},
    {"policy": "random", "test margin": MARGIN[test, random_arms[test]].mean(), "test click": CLICK[test, random_arms[test]].mean()},
])
print(summary.round(3).to_string(index=False))
assert summary.loc[0, "test margin"] > summary.loc[2, "test margin"]
assert summary.loc[0, "test margin"] > summary.loc[1, "test margin"]
assert summary.loc[1, "test click"] > summary.loc[0, "test click"]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.cumsum(margin_reward - margin_reward.mean()), label="margin policy, centered")
ax.plot(np.cumsum(click_reward - click_reward.mean()), label="click policy, centered")
ax.legend()
ax.set_title("Centered cumulative reward of each training signal")
ax.set_xlabel("round")
plt.tight_layout()
plt.show()


## Guided exercise: offline evaluation

A uniform logging policy collects one reward per round. IPS of that same policy must equal the mean logged reward. The deterministic target is the frozen margin LinUCB. Doubly robust estimation fits a ridge reward model per arm on the log and adds the logged residual only when the logged arm matches the target. Clipping the importance weight is reported so you can see the bias it introduces. The clip is not free.


In [ ]:
log_arm = rng.integers(0, len(ARMS), size=N)
propensity = np.full(N, 1.0 / len(ARMS))
log_reward = np.array([MARGIN[t, log_arm[t]] + float(rng.normal(0, 0.15)) for t in range(N)])

def ips_stochastic(rewards, pi, p_b):
    return float(np.mean(rewards * pi / p_b))

def ips_deterministic(rewards, logged, target, p_b, clip=None):
    weights = np.where(logged == target, 1.0 / p_b, 0.0)
    if clip is not None:
        weights = np.minimum(weights, clip)
    return float(np.mean(weights * rewards)), float(weights.max())

target = np.array([margin_policy.choose(X[t]) for t in range(N)])
uniform_pi = np.full(N, 1.0 / len(ARMS))
ips_log = ips_stochastic(log_reward, uniform_pi, propensity)
ips_target, max_w = ips_deterministic(log_reward, log_arm, target, propensity)
ips_clipped, max_w_clipped = ips_deterministic(log_reward, log_arm, target, propensity, clip=3.0)

models = []
for arm in ARMS:
    mask = log_arm == arm
    model = Ridge(alpha=1.0)
    model.fit(X[mask], log_reward[mask])
    models.append(model)
r_hat = np.zeros((N, len(ARMS)))
for arm, model in enumerate(models):
    r_hat[:, arm] = model.predict(X)
r_logged = r_hat[np.arange(N), log_arm]
r_target = r_hat[np.arange(N), target]
weights = np.where(log_arm == target, 1.0 / propensity, 0.0)
dr = float(np.mean(r_target + weights * (log_reward - r_logged)))

# On-policy value of the target, using the known expected margin. This is the number IPS is aiming at.
truth = float(MARGIN[np.arange(N), target].mean())
print(pd.DataFrame([
    {"estimate": "logging IPS (should equal mean reward)", "value": ips_log},
    {"estimate": "mean logged reward", "value": float(log_reward.mean())},
    {"estimate": "target IPS", "value": ips_target},
    {"estimate": "target IPS clipped at 3", "value": ips_clipped},
    {"estimate": "target doubly robust", "value": dr},
    {"estimate": "target expected margin", "value": truth},
]).round(3).to_string(index=False))
assert abs(ips_log - log_reward.mean()) < 1e-9
assert max_w_clipped <= 3.0 + 1e-9
assert abs(dr - truth) < abs(float(log_reward.mean()) - truth)
print("LAB OK")


## Your turn

The uncertainty bonus uses `alpha`. Set `alpha` to 0, which is greedy ridge regression with no exploration, and compare test margin with the `alpha=0.6` policy above.

## Solution


In [ ]:
_, greedy_arms, _ = rollout(MARGIN, alpha=0.0)
greedy_margin = float(MARGIN[test, greedy_arms[test]].mean())
explore_margin = float(summary.loc[0, "test margin"])
print("alpha 0", round(greedy_margin, 3), "alpha 0.6", round(explore_margin, 3))
assert np.isfinite(greedy_margin)


## Beyond the course

The click-trained policy and the margin-trained policy are evaluated on both rewards. The experiment is whether the click policy's higher click rate is bought by giving away margin. The assertion is the business one: on this simulator, optimizing click lowers margin relative to optimizing margin.


In [ ]:
print(summary.round(3).to_string(index=False))
click_trained_margin = float(summary.loc[1, "test margin"])
margin_trained_margin = float(summary.loc[0, "test margin"])
assert margin_trained_margin > click_trained_margin


## Research questions

1. Doubly robust estimation was closer to the known margin than the raw mean of the log. Which term would you trust if the reward model were fit on a different offer catalog than the log?
2. The propensities are known because the logger was uniform. What bias appears if a production logger rounds propensities to two digits?
3. Where is the delay in a real next-best-offer reward (a click now, a cancellation in three months), and which of these estimators ignores it?

## Further reading

- Li, Chu, Langford, and Schapire, *A Contextual-Bandit Approach to Personalized News Article Recommendation* (WWW 2010).
- Dudík, Langford, and Li, *Doubly Robust Policy Evaluation and Learning* (ICML 2011).
- Horvitz and Thompson, *A Generalization of Sampling Without Replacement from a Finite Universe* (1952).
- Christiano et al., *Deep Reinforcement Learning from Human Preferences* (2017), for the same reward-design problem in language models. The offer simulator is the small version.
